# Task 4 — Stage 1: multilingual transcription and speaker turns

This Colab notebook runs open-source models locally in the Colab GPU runtime. It does not call a paid speech API. It processes the four supplied MP3s, produces draft transcripts with timestamps, and assigns anonymous speaker IDs.

**Current scope:** Arabic, Bengali, Malayalam, and Urdu audio; three speakers expected per recording. The diarizer can label `SPEAKER_00` etc., but it cannot know which one is the doctor, patient, or nurse. Review the turns and fill the `role` column manually. Keep the draft text until a fluent reviewer checks it.

This first stage does not generate clinical reports or claim transcription accuracy. Accuracy scores require verified reference transcripts.


## 1. Select the runtime and install

In Colab, choose **Runtime → Change runtime type → T4 GPU**. Then run this cell. WhisperX provides the draft transcription and speaker diarization. For a single-language run, upload just that MP3 in the upload cell; for a batch run, upload all four. The pyannote diarization stage requires accepting Hugging Face model terms and using a read token.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install whisperx pandas


In [ ]:
import gc, json, os, zipfile, warnings
from pathlib import Path
import pandas as pd
import torch
from google.colab import files

assert torch.cuda.is_available(), "Choose a GPU runtime: Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

# Upload one language to test, or all four for a batch run. Keep these filenames.
uploaded = files.upload()
expected = {
    "arabic.mp3": ("Arabic", "ar"),
    "bengali.mp3": ("Bengali", "bn"),
    "malayalam.mp3": ("Malayalam", "ml"),
    "urdu.mp3": ("Urdu", "ur"),
}
found = {name.lower(): name for name in uploaded}
AUDIO_PATHS = {lang: Path(found[filename]) for filename, (lang, _) in expected.items() if filename in found}
LANG_CODES = {lang: code for filename, (lang, code) in expected.items() if filename in found}
assert AUDIO_PATHS, "Upload at least one supported file: arabic.mp3, bengali.mp3, malayalam.mp3, or urdu.mp3."
OUTPUT_DIR = Path("/content/task4_stage1_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Ready:", {lang: str(path) for lang, path in AUDIO_PATHS.items()})

## 2. Transcribe each language

The language is set from each filename to reduce language confusion. This cell loads one model and processes the four clips sequentially. If the T4 runs out of memory, lower `BATCH_SIZE` to `2`; if needed, switch `COMPUTE_TYPE` to `int8` (which can affect accuracy).


In [ ]:
import whisperx

ASR_MODEL = "large-v3-turbo"
BATCH_SIZE = 4
COMPUTE_TYPE = "float16"
DEVICE = "cuda"

torch.cuda.reset_peak_memory_stats()
asr_model = whisperx.load_model(ASR_MODEL, DEVICE, compute_type=COMPUTE_TYPE)
asr_results = {}
for language, audio_path in AUDIO_PATHS.items():
    print(f"Transcribing {language}: {audio_path.name}")
    audio = whisperx.load_audio(str(audio_path))
    result = asr_model.transcribe(audio, batch_size=BATCH_SIZE, language=LANG_CODES[language])
    asr_results[language] = result
    print("Detected/result language:", result.get("language"))
    print("Draft:", " ".join(s.get("text", "").strip() for s in result.get("segments", []))[:1000])
    del audio

peak_asr_gib = torch.cuda.max_memory_allocated() / 1024**3
print("WhisperX peak allocated VRAM (GiB):", round(peak_asr_gib, 2))
del asr_model
gc.collect(); torch.cuda.empty_cache()


## 3. Add speaker IDs

Diarization uses the open `pyannote/speaker-diarization-community-1` model through WhisperX. Before running this cell, open its [Hugging Face model page](https://huggingface.co/pyannote/speaker-diarization-community-1), sign in, and accept its usage conditions. Create a Hugging Face **read** token. Enter it into Colab's hidden prompt below; do not paste the token into this notebook or share it.

The script expects three speakers as in the source script. It exports anonymous IDs only. You will assign doctor/patient/nurse after listening.


In [ ]:
from getpass import getpass
from whisperx.diarize import DiarizationPipeline

hf_token = getpass("Hugging Face read token (hidden input): ")
diarize_model = DiarizationPipeline(token=hf_token, device=DEVICE)
del hf_token

def overlap_seconds(a0, a1, b0, b1):
    return max(0.0, min(float(a1), float(b1)) - max(float(a0), float(b0)))

all_rows = []
all_json = {}
for language, audio_path in AUDIO_PATHS.items():
    print(f"Diarizing {language}: {audio_path.name}")
    audio = whisperx.load_audio(str(audio_path))
    diar = diarize_model(audio, min_speakers=3, max_speakers=3)
    # WhisperX returns a table with start/end/speaker columns.
    diar_rows = diar.to_dict("records")
    output_segments = []
    for seg in asr_results[language].get("segments", []):
        start, end = float(seg.get("start", 0)), float(seg.get("end", 0))
        overlaps = {}
        for d in diar_rows:
            speaker = d.get("speaker")
            if speaker is None:
                continue
            duration = overlap_seconds(start, end, d.get("start", 0), d.get("end", 0))
            overlaps[speaker] = overlaps.get(speaker, 0.0) + duration
        speaker_id = max(overlaps, key=overlaps.get) if overlaps and max(overlaps.values()) > 0 else "UNKNOWN"
        row = {
            "sample_id": audio_path.stem, "language": language,
            "start_seconds": round(start, 3), "end_seconds": round(end, 3),
            "speaker_id": speaker_id, "role": "",
            "draft_transcript": seg.get("text", "").strip(),
            "reviewed_transcript": "",
        }
        all_rows.append(row)
        output_segments.append(row)
    all_json[language] = {"audio_file": audio_path.name, "language": language, "asr_model": ASR_MODEL, "segments": output_segments}
    print("Speaker IDs found:", sorted({r["speaker_id"] for r in output_segments}))
    del audio, diar
    gc.collect(); torch.cuda.empty_cache()

segments_df = pd.DataFrame(all_rows)
print("\nDiarized segment preview:")
display(segments_df.head(20))


## 4. Save the draft for review

Download the CSV and JSON bundle. In the CSV, listen to the audio and fill in `role` as `doctor`, `patient`, or `nurse`; correct `reviewed_transcript` only after checking the language. Speaker IDs may be numbered differently in each file. Do not use the automatic speaker labels as proof of a role.

For a fair ASR benchmark, enter a fluent-reviewed full reference transcript per recording in a separate file later. The source text in the shared chat is useful as a draft reference, but machine-generated translations should be checked before they count as ground truth.


In [ ]:
csv_path = OUTPUT_DIR / "speaker_transcript_review.csv"
json_path = OUTPUT_DIR / "speaker_transcripts.json"
segments_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
json_path.write_text(json.dumps(all_json, ensure_ascii=False, indent=2), encoding="utf-8")

zip_path = Path("/content/task4_stage1_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(csv_path, arcname=csv_path.name)
    zf.write(json_path, arcname=json_path.name)
print("Saved:", csv_path, json_path, zip_path)
files.download(str(zip_path))


## What to do after this run

1. Review the speaker turns and transcript with a fluent speaker; correct the CSV.
2. Confirm which anonymous speaker is the doctor, patient, and nurse for each audio.
3. Add the verified reference transcript. Then compare ASR systems with per-language WER/CER and manually check medical terms and numbers.
4. Only after transcript correction, pass the transcript to the clinical extraction models and create separate English and same-language reports.

WhisperX documents that diarization is imperfect and overlapping speech is difficult. Treat its output as a draft requiring review ([official WhisperX instructions and limitations](https://github.com/m-bain/whisperX)).


## 5. Compare with official OpenAI Whisper Large-v3

This is a separate full-model run (not WhisperX's `large-v3-turbo`). It produces another transcript draft and timing/resource metadata for the four uploaded files. Script checks are not accuracy scores. Do not use these drafts as final medical documentation before fluent review.


In [ ]:
# Official OpenAI Whisper multilingual large-v3; run after the audio upload cell.
# This cell is compatible with the existing AUDIO_PATHS and LANG_CODES dictionaries.
!pip -q install openai-whisper

import gc, importlib.metadata, json, time
import pandas as pd
import torch
import whisper

assert torch.cuda.is_available(), "Select a GPU runtime (for example, T4)."
print("GPU:", torch.cuda.get_device_name(0))
print("Whisper package:", importlib.metadata.version("openai-whisper"))

torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
whisper_model = whisper.load_model("large-v3", device="cuda")
model_load_seconds = time.perf_counter() - t0
print(f"Loaded openai/whisper large-v3 in {model_load_seconds:.1f} seconds")

whisper_results = {}
rows = []
for language, audio_path in AUDIO_PATHS.items():
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    result = whisper_model.transcribe(
        str(audio_path), language=LANG_CODES[language], task="transcribe",
        fp16=True, verbose=False, temperature=0, condition_on_previous_text=False
    )
    elapsed = time.perf_counter() - started
    peak_gib = torch.cuda.max_memory_allocated() / 1024**3
    transcript = result.get("text", "").strip()
    whisper_results[language] = {
        "audio_file": audio_path.name, "language_code": LANG_CODES[language],
        "detected_language": result.get("language"), "transcript": transcript,
        "segments": result.get("segments", []),
        "inference_seconds": elapsed, "peak_allocated_gpu_gib": peak_gib,
    }
    rows.append({
        "sample_id": language.lower(), "language": language,
        "model": "openai/whisper-large-v3",
        "detected_language": result.get("language"),
        "inference_seconds": elapsed, "peak_allocated_gpu_gib": peak_gib,
        "transcript": transcript,
    })
    print(f"\n{language} | {elapsed:.1f}s | peak GPU {peak_gib:.2f} GiB")
    print(transcript[:1200])

whisper_json = OUTPUT_DIR / "whisper_large_v3_results.json"
whisper_csv = OUTPUT_DIR / "whisper_large_v3_results.csv"
whisper_json.write_text(json.dumps({
    "model": "openai/whisper-large-v3",
    "hardware": torch.cuda.get_device_name(0),
    "package_version": importlib.metadata.version("openai-whisper"),
    "model_load_seconds": model_load_seconds,
    "results": whisper_results,
    "total_peak_allocated_gpu_gib": max(r["peak_allocated_gpu_gib"] for r in whisper_results.values()),
}, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame(rows).to_csv(whisper_csv, index=False, encoding="utf-8-sig")
print("Saved:", whisper_json, whisper_csv)


## 6. Translate the audio into English on the T4 GPU

Run this after the audio upload cell. It translates directly from the regional audio with OpenAI Whisper `large-v3`. It reuses the model from the comparison cell if loaded; otherwise it loads the model on the T4. For a quicker English-only run, skip the separate WhisperX transcription cells and run this cell after upload. To add anonymous speaker IDs, run the diarization cell first; those IDs are not doctor/patient/nurse role labels.

This produces a machine-generated draft. Check medical terms, names, numbers, speaker turns, and meaning against the audio before using it.

In [ ]:
!pip -q install openai-whisper
import gc, json, time, zipfile
from pathlib import Path
import pandas as pd
import torch
import whisper

assert torch.cuda.is_available(), "Select Runtime → Change runtime type → T4 GPU."
if "whisper_model" not in globals():
    print("Loading Whisper large-v3 onto the T4. This is only needed if the model cell above was not run in this session.")
    whisper_model = whisper.load_model("large-v3", device="cuda")

def _speaker_for_time(start, end, diarized_segments):
    overlap_by_speaker = {}
    for source in diarized_segments:
        speaker = source.get("speaker_id")
        if not speaker:
            continue
        overlap = max(0.0, min(float(end), float(source.get("end_seconds", 0))) - max(float(start), float(source.get("start_seconds", 0))))
        if overlap > 0:
            overlap_by_speaker[speaker] = overlap_by_speaker.get(speaker, 0.0) + overlap
    return max(overlap_by_speaker, key=overlap_by_speaker.get) if overlap_by_speaker else "UNKNOWN"

english_translation_results = {}
translation_rows = []
translation_txt_files = []
torch.cuda.reset_peak_memory_stats()
for language, audio_path in AUDIO_PATHS.items():
    print(f"Translating {language} audio to English: {audio_path.name}")
    started = time.perf_counter()
    translated = whisper_model.transcribe(
        str(audio_path), language=LANG_CODES[language], task="translate",
        fp16=True, verbose=False, temperature=0, condition_on_previous_text=False
    )
    elapsed = time.perf_counter() - started
    diarized_segments = globals().get("all_json", {}).get(language, {}).get("segments", [])
    output_segments = []
    for segment in translated.get("segments", []):
        start, end = float(segment.get("start", 0)), float(segment.get("end", 0))
        speaker_id = _speaker_for_time(start, end, diarized_segments)
        item = {"start": start, "end": end, "speaker_id": speaker_id, "text": str(segment.get("text", "")).strip()}
        output_segments.append(item)
        translation_rows.append({"language": language, "audio_file": audio_path.name, **item})
    transcript = "\n".join((f"[{s['speaker_id']}] " if s["speaker_id"] != "UNKNOWN" else "") + s["text"] for s in output_segments if s["text"])
    english_translation_results[language] = {
        "audio_file": audio_path.name, "source_language": language,
        "model": "openai/whisper-large-v3", "task": "translate",
        "translation_seconds": elapsed, "transcript": transcript,
        "segments": output_segments,
        "speaker_ids_are_anonymous_and_unverified": True,
    }
    txt_path = OUTPUT_DIR / f"{language.lower()}_english_translation.txt"
    txt_path.write_text(transcript + "\n", encoding="utf-8")
    translation_txt_files.append(txt_path)
    print(f"Finished in {elapsed:.1f}s; preview:")
    print(transcript[:1200] or "[No English text returned]")

translation_json = OUTPUT_DIR / "large_v3_english_translations.json"
translation_csv = OUTPUT_DIR / "large_v3_english_translations.csv"
translation_json.write_text(json.dumps({
    "model": "openai/whisper-large-v3",
    "hardware": torch.cuda.get_device_name(0),
    "results": english_translation_results,
    "note": "Machine translation draft; verify medical terms and speaker turns against audio."
}, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame(translation_rows).to_csv(translation_csv, index=False, encoding="utf-8-sig")
translation_zip = Path("/content/large_v3_english_translations.zip")
with zipfile.ZipFile(translation_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(translation_json, arcname=translation_json.name)
    archive.write(translation_csv, arcname=translation_csv.name)
    for txt_path in translation_txt_files:
        archive.write(txt_path, arcname=txt_path.name)
print("Saved English translation drafts:", translation_zip)
files.download(str(translation_zip))